<a href="https://colab.research.google.com/github/SyameimaruKoa/Minecraft-mod-AutoTranslation-tool/blob/main/run_colab_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minecraft Mod Auto Translator — Google Colab（LM Link）

ColabでLM Studio（llmster）を動かし、LM Link経由でPCと接続するのじゃ。認証情報は`dbv4-wd14-tagger`と共通のGoogleドライブフォルダーへ**暗号化保存**する。モデルはpfnet公式のGGUFファイルだけを使用し、既定は8B／Q4_K_Mじゃ。

## 1. サーバー設定

`dbv4-wd14-tagger`のColab版と同じ`USE_GOOGLE_DRIVE`と`DRIVE_FOLDER_NAME`を使用する。既定の保存先は `MyDrive/.colab-secret/` じゃ。

In [ ]:
#@title ⚙️ サーバー設定 (Configuration)
#@markdown dbv4-wd14-taggerと同じGoogleドライブ保存先設定じゃ。

USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = ".colab-secret" #@param {type:"string"}

CONFIG_DRIVE_FOLDER = DRIVE_FOLDER_NAME.strip("/ ") if DRIVE_FOLDER_NAME else ".colab-secret"
if (
    CONFIG_DRIVE_FOLDER in (".", "..")
    or "/" in CONFIG_DRIVE_FOLDER
    or "\\" in CONFIG_DRIVE_FOLDER
):
    raise ValueError("Googleドライブの保存先は単一フォルダー名を指定するのじゃ")
print(
    "[INFO] Googleドライブ保存: "
    + (f"有効 (MyDrive/{CONFIG_DRIVE_FOLDER})" if USE_GOOGLE_DRIVE else "無効")
)

## 2. LM Studioのインストール

先にインストールし、次のセルで認証情報を復元してからデーモンを起動する。

In [ ]:
import os
import subprocess
from pathlib import Path

subprocess.run(["pkill", "-f", "llmster"], check=False)
subprocess.run(["pkill", "-f", "lmstudio"], check=False)
subprocess.run(["bash", "-c", "curl -fsSL https://lmstudio.ai/install.sh | bash"], check=True)
os.environ["PATH"] = str(Path.home() / ".lmstudio" / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run(["lms", "bootstrap"], check=True)
subprocess.run(["sync"], check=True)
print("LM Studioのセットアップが完了したのじゃ。")

## 3. Googleドライブへのログイン状態保存

DBV4のTailscaleと同様、LM Studioのログイン状態をGoogleドライブの `MyDrive/.colab-secret/lmstudio/` 以下へ保存・自動復元する。パスフレーズの入力やColabシークレットは不要じゃ。保存対象は `credentials` と `hub` の認証状態のみで、内部デーモンキーやモデル本体は保存しない。

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

LM_HOME = Path.home() / ".lmstudio"
DRIVE_AUTH_DIR = None
AUTH_DIRS = ("credentials", "hub")

if USE_GOOGLE_DRIVE:
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    DRIVE_AUTH_DIR = Path("/content/drive/MyDrive") / CONFIG_DRIVE_FOLDER / "lmstudio"

def copy_login_files(source, destination):
    copied = 0
    for directory in AUTH_DIRS:
        root = source / directory
        if not root.is_dir() or root.is_symlink():
            continue
        for original in root.rglob("*"):
            if not original.is_file() or original.is_symlink():
                continue
            relative = original.relative_to(source)
            target = destination / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, target)
            os.chmod(target, 0o600)
            copied += 1
    return copied

def restore_login():
    if not USE_GOOGLE_DRIVE:
        return False
    restored = copy_login_files(DRIVE_AUTH_DIR, LM_HOME)
    if restored:
        print(f"[INFO] GoogleドライブからLM Link認証情報を復元: {restored}ファイル")
    else:
        print("[INFO] 保存済みのLM Link認証情報なし。初回ログインが必要じゃ。")
    return bool(restored)

def backup_login():
    if not USE_GOOGLE_DRIVE:
        return
    saved = copy_login_files(LM_HOME, DRIVE_AUTH_DIR)
    if not saved:
        raise RuntimeError("LM Link認証ファイルが見つからないため、Googleドライブへ保存できぬ")
    print(f"[INFO] LM Link認証情報をGoogleドライブに保存: {saved}ファイル")

auth_restored = restore_login()

## 4. LM Linkのログインと接続

保存済み認証情報が有効なら再ログインせずに接続する。初回またはログイン期限切れなら `lms login` の認証URLをブラウザで開くのじゃ。ログイン後に認証情報を暗号化保存する。

In [ ]:
import subprocess
import time

subprocess.run(["lms", "daemon", "up"], check=True)
time.sleep(3)
whoami = subprocess.run(
    ["lms", "whoami"], stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL, check=False, timeout=30
)
if whoami.returncode != 0:
    print("LM Studioのログインが必要じゃ。表示される認証URLで承認するのじゃ。")
    subprocess.run(["lms", "login"], check=True)
else:
    print("保存済みのLM Studioログインが有効じゃ。")
subprocess.run(["lms", "link", "enable"], check=True)
backup_login()
print("LM Linkが有効になったのじゃ。")

## 5. PLaMo 3 Translateモデルのダウンロードとロード

既定は**8B／Q4_K_M**。0.8B・2B・8B・31Bからサイズを選び、pfnetが公式に配布しているGGUFの量子化形式だけ指定する。`lms get`の検索処理を使わず、PLaMo公式のGGUFファイルを直接取得するため、問題のコマンドは実行しない。中断したダウンロードは途中から再開する。Colabランタイムを破棄するとモデルファイルは消えるが、Googleドライブの保存済み認証情報は残る。31BはGPUメモリ不足でロードできない場合があるのじゃ。

In [ ]:
import os
import subprocess
from pathlib import Path

plamo_size = "8b" # @param ["0.8b", "2b", "8b", "31b"]
quantization = "Q4_K_M" # @param ["Q4_K_M", "Q3_K_M", "Q2_K", "Q4_0"]

OFFICIAL_GGUF_FORMATS = {
    "0.8b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "2b": ("Q4_K_M", "Q4_0", "Q3_K_M", "Q2_K"),
    "8b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
    "31b": ("Q4_K_M", "Q3_K_M", "Q2_K"),
}
if plamo_size not in OFFICIAL_GGUF_FORMATS:
    raise ValueError("未対応のPLaMoサイズじゃ")
if quantization not in OFFICIAL_GGUF_FORMATS[plamo_size]:
    raise ValueError(f"pfnet公式で配布されていない組み合わせじゃ: {plamo_size}/{quantization}")

repo_name = f"plamo-3-translate-{plamo_size}-GGUF"
filename = f"plamo-3-translate-{plamo_size}-{quantization}.gguf"
model_dir = Path.home() / ".lmstudio" / "models" / "pfnet" / repo_name
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / filename
partial_path = model_dir / (filename + ".part")
model_id = f"plamo-3-translate-{plamo_size}-colab"
url = f"https://huggingface.co/pfnet/{repo_name}/resolve/main/{filename}?download=true"

if not model_path.exists() or model_path.stat().st_size < 100_000_000:
    print(f"公式GGUFを取得: pfnet/{repo_name}/{filename}", flush=True)
    subprocess.run(
        ["curl", "--fail", "--location", "--retry", "3", "--retry-delay", "3",
         "--continue-at", "-", "--output", str(partial_path), url],
        check=True
    )
    if partial_path.stat().st_size < 100_000_000:
        raise RuntimeError("GGUFのダウンロード結果が小さすぎるのじゃ")
    os.replace(partial_path, model_path)
else:
    print(f"ダウンロード済みGGUFを再利用: {filename}", flush=True)

subprocess.run(["lms", "unload", "--all"], check=True)
subprocess.run(
    ["lms", "load", str(model_path), "--gpu", "max", "--identifier", model_id],
    check=True
)
subprocess.run(["lms", "server", "start", "--port", "1234"], check=True)
subprocess.run(["lms", "ps"], check=True)
print(f"ロード完了: {model_id}（LM Linkから利用可能）", flush=True)